In [12]:
import kagglehub
base_path = kagglehub.dataset_download("naveedhn/amazon-product-review-spam-and-non-spam")
print(base_path)

C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1


In [13]:
import os
import pandas as pd

print("Dataset path:", base_path)
print("Files inside:", os.listdir(base_path))


Dataset path: C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1
Files inside: ['Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry', 'Electronics', 'Home_and_Kitchen', 'part.json', 'separate.json', 'Sports_and_Outdoors', 'Toys_and_Games']


In [14]:
import os

# Ana klasörde neler var
print("Ana klasör:", base_path)
print(os.listdir(base_path))

# Örneğin ilk klasörün içine bakalım
first_folder = os.path.join(base_path, "Electronics")
print("\Electronics içeriği:")
print(os.listdir(first_folder))


Ana klasör: C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1
['Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry', 'Electronics', 'Home_and_Kitchen', 'part.json', 'separate.json', 'Sports_and_Outdoors', 'Toys_and_Games']
\Electronics içeriği:
['Electronics.json']


<>:9: SyntaxWarning: invalid escape sequence '\E'
<>:9: SyntaxWarning: invalid escape sequence '\E'
C:\Users\aadil\AppData\Local\Temp\ipykernel_21024\2652339389.py:9: SyntaxWarning: invalid escape sequence '\E'
  print("\Electronics içeriği:")


In [18]:
import os

electronics_path = os.path.join(base_path, "Electronics")
print("Electronics klasörü içeriği:")
print(os.listdir(electronics_path))


Electronics klasörü içeriği:
['Electronics.json']


In [21]:
import pandas as pd
import os

file_path = os.path.join(electronics_path, os.listdir(electronics_path)[0])  # ilk dosyayı seç
print("Seçilen dosya:", file_path)

# Dosya boyutunu kontrol et
file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB cinsinden
print(f"Dosya boyutu: {file_size:.2f} MB")

# Çok büyük dosyalar için güvenli okuma
if file_size > 100:  # 100 MB'dan büyükse
    print("Dosya çok büyük, ilk 1000 satırı okuyorum...")
    df_electronics = pd.read_json(file_path, lines=True, nrows=1000)
else:
    print("Dosya boyutu uygun, tamamını okuyorum...")
    df_electronics = pd.read_json(file_path, lines=True)

print(f"Yüklenen veri shape: {df_electronics.shape}")
print("İlk 5 satır:")
df_electronics.head(2)

Seçilen dosya: C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1\Electronics\Electronics.json
Dosya boyutu: 5492.40 MB
Dosya çok büyük, ilk 1000 satırı okuyorum...
Yüklenen veri shape: (1000, 12)
İlk 5 satır:


,_id,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,category,class
0,{'$oid': '5a13242d741a2384e88f3c11'},A2WNBOD3WNDNKT,0439886341,JAL,"[1, 1]",Some of the functions did not work properly. ...,3,Disappointing,1374451200,"07 22, 2013",Electronics,0
1,{'$oid': '5a13242d741a2384e88f3c0f'},AKM1MP6P0OYPR,0132793040,"Vicki Gibson ""momo4""","[1, 1]",Corey Barker does a great job of explaining Bl...,5,Very thorough,1365811200,"04 13, 2013",Electronics,1


## Adım 1: Dosyaları bul ve örnek dosya seç
Bütün JSON dosyalarını `base_path` altında geziyoruz, ilk örnek dosyayı seçip boyutunu görüyoruz.

In [22]:
import os, glob

base = base_path if 'base_path' in globals() else os.getcwd()
json_files = sorted(glob.glob(os.path.join(base, '**', '*.json'), recursive=True))
print(f'JSON dosya sayısı: {len(json_files)}')
for p in json_files[:10]:
    try:
        sz = os.path.getsize(p)/(1024*1024*1024)
        print(f'- {p}  ({sz:.2f} GB)')
    except Exception:
        print(f'- {p}')

if not json_files:
    raise SystemExit('JSON dosyası bulunamadı. İndirme adımını kontrol edin.')
sample_path = json_files[0]
print('Seçilen örnek dosya:', sample_path)

JSON dosya sayısı: 10
- C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1\Cell_Phones_and_Accessories\Cell_Phones_and_Accessories.json  (1.87 GB)
- C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1\Clothing_Shoes_and_Jewelry\Clothing_Shoes_and_Jewelry.json  (2.99 GB)
- C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1\Electronics\Electronics.json  (5.36 GB)
- C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1\Home_and_Kitchen\Home_and_Kitchen.json  (2.50 GB)
- C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1\Sports_and_Outdoors\Sports_and_Outdoors.json  (1.86 GB)
- C:\Users\aadil\.cache\kagglehub\datasets\naveedhn\amazon-product-review-spam-and-non-spam\versions\1\Toys_and_Games\Toys_and_Games.json  (1.20 GB)
- C:\Users\aadil\.cache\ka

## Adım 2: Küçük bir örnek oku (güvenli)
Önce `lines=True` (NDJSON) deneriz, olmazsa normal JSON olarak sınırlı satır okuruz.

In [23]:
import pandas as pd

def read_json_sample(path: str, nrows: int = 10000) -> pd.DataFrame:
    try:
        return pd.read_json(path, lines=True, nrows=nrows)
    except ValueError:
        try:
            return pd.read_json(path, nrows=nrows)
        except Exception as e:
            print('Okuma hatası:', e)
            return pd.DataFrame()

df_sample = read_json_sample(sample_path, nrows=10000)
print('Örnek shape:', df_sample.shape)
print('Sütunlar (ilk 30):', list(df_sample.columns)[:30])
df_sample.head(3)

Örnek shape: (10000, 12)
Sütunlar (ilk 30): ['_id', 'reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime', 'category', 'class']


,_id,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,category,class
0,{'$oid': '5a1321d5741a2384e802c552'},A3HVRXV0LVJN7,0110400550,BiancaNicole,"[4, 4]",Best phone case ever . Everywhere I go I get a...,5,A++++,1358035200,"01 13, 2013",Cell_Phones_and_Accessories,1
1,{'$oid': '5a1321d5741a2384e802c557'},A1BJGDS0L1IO6I,0110400550,"cf ""t""","[0, 3]",ITEM NOT SENT from Blue Top Company in Hong Ko...,1,ITEM NOT SENT!!,1359504000,"01 30, 2013",Cell_Phones_and_Accessories,0
2,{'$oid': '5a1321d5741a2384e802c550'},A1YX2RBMS1L9L,0110400550,Andrea Busch,"[0, 0]",Saw this same case at a theme park store for 2...,5,Great product,1353542400,"11 22, 2012",Cell_Phones_and_Accessories,1


## Adım 3: Tip normalizasyonu ve yararlı alanların türetilmesi
- class → label (int)
- unixReviewTime → datetime
- helpful → helpful_up, helpful_tot, helpful_ratio
- reviewText + summary → text

In [31]:
import numpy as np

df_sample = df_sample.copy()
# Etiket
if 'class' in df_sample.columns:
    df_sample['label'] = pd.to_numeric(df_sample['class'], errors='coerce').fillna(0).astype('int8')
elif 'label' in df_sample.columns:
    df_sample['label'] = pd.to_numeric(df_sample['label'], errors='coerce').fillna(0).astype('int8')

# Tarih
if 'unixReviewTime' in df_sample.columns:
    df_sample['review_dt'] = pd.to_datetime(df_sample['unixReviewTime'], unit='s', errors='coerce')

# Helpful parçalama
if 'helpful' in df_sample.columns:
    up = df_sample['helpful'].apply(lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) > 0 else np.nan)
    tot = df_sample['helpful'].apply(lambda x: x[1] if isinstance(x, (list, tuple)) and len(x) > 1 else np.nan)
    df_sample['helpful_up'] = up.astype('float32')
    df_sample['helpful_tot'] = tot.astype('float32')
    df_sample['helpful_ratio'] = (df_sample['helpful_up'] / df_sample['helpful_tot'].replace(0, np.nan)).astype('float32')

# Metin birleştirme
rt = df_sample['reviewText'] if 'reviewText' in df_sample.columns else ''
sm = df_sample['summary'] if 'summary' in df_sample.columns else ''
df_sample['text'] = (rt.fillna('') + ' ' + sm.fillna('')).str.strip()

print('Dönüşüm sonrası sütunlar (ilk 30):', list(df_sample.columns)[:30])
df_sample[['asin' if 'asin' in df_sample.columns else df_sample.columns[0], 'label' if 'label' in df_sample.columns else df_sample.columns[1], 'review_dt' if 'review_dt' in df_sample.columns else df_sample.columns[2]]].head(3)

Dönüşüm sonrası sütunlar (ilk 30): ['_id', 'reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime', 'category', 'class', 'text', 'text_clean', 'char_len', 'word_len', 'bang_cnt', 'qmark_cnt', 'label', 'review_dt', 'helpful_up', 'helpful_tot', 'helpful_ratio']


,asin,label,review_dt
0,0110400550,1,2013-01-13
1,0110400550,0,2013-01-30
2,0110400550,1,2012-11-22


## Adım 4: Metin temizliği ve basit özellikler
Küçük bir temizleme uygular ve uzunluk/işaret sayıları gibi basit özellikler ekleriz.

In [32]:
import re

# 'text' kolonu yoksa oluşturalım (reviewText + summary)
if 'text' not in df_sample.columns:
    rt = df_sample['reviewText'] if 'reviewText' in df_sample.columns else ''
    sm = df_sample['summary'] if 'summary' in df_sample.columns else ''
    df_sample['text'] = (rt.fillna('') + ' ' + sm.fillna('')).str.strip()


def clean_text_basic(s: str) -> str:
    if not isinstance(s, str):
        return ''
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'<[^>]+>', ' ', s)
    s = re.sub(r'[\r\n\t]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# Temiz metin ve basit özellikler
df_sample['text_clean'] = df_sample['text'].fillna('').map(clean_text_basic)

df_sample['char_len'] = df_sample['text_clean'].str.len().astype('int32')
df_sample['word_len'] = df_sample['text_clean'].str.split().map(len).astype('int32')
# .str.count regex kullanır; ? işaretini saymak için kaçış gerekir
df_sample['bang_cnt'] = df_sample['text_clean'].str.count(r'!').astype('int16')
df_sample['qmark_cnt'] = df_sample['text_clean'].str.count(r'\?').astype('int16')

# Çok kısa metinleri ele (isteğe bağlı)
before = len(df_sample)
df_sample = df_sample[df_sample['word_len'] >= 3].reset_index(drop=True)
removed = before - len(df_sample)

print('Temizlik sonrası shape:', df_sample.shape, '| Kaldırılan kısa kayıt:', removed)
print(df_sample[['text_clean','word_len','bang_cnt','qmark_cnt']].head(3))

Temizlik sonrası shape: (10000, 23) | Kaldırılan kısa kayıt: 0
                                          text_clean  word_len  bang_cnt  \
0  best phone case ever . everywhere i go i get a...        24         0   
1  item not sent from blue top company in hong ko...        32         4   
2  saw this same case at a theme park store for 2...        23         0   

   qmark_cnt  
0          0  
1          0  
2          0  


## Adım 5: Önizleme ve (isteğe bağlı) kayıt
Önemli sütunları gösterir ve küçük bir örneği Parquet olarak kaydederiz.

In [33]:
keep_cols = [c for c in [
    'asin','reviewerID','category','overall','label',
    'review_dt','helpful_up','helpful_tot','helpful_ratio',
    'word_len','char_len','bang_cnt','qmark_cnt','text_clean'
] if c in df_sample.columns]

print('Önizleme sütunları:', keep_cols)
print(df_sample[keep_cols].head(5))

out_path = os.path.join(os.getcwd(), 'preprocessed_sample.parquet')
try:
    df_sample[keep_cols].to_parquet(out_path, index=False)
    print('Kaydedildi:', out_path)
except Exception as e:
    print('Parquet kaydı başarısız olabilir (pyarrow gerekli). Hata:', e)

Önizleme sütunları: ['asin', 'reviewerID', 'category', 'overall', 'label', 'review_dt', 'helpful_up', 'helpful_tot', 'helpful_ratio', 'word_len', 'char_len', 'bang_cnt', 'qmark_cnt', 'text_clean']
         asin      reviewerID                     category  overall  label  \
0  0110400550   A3HVRXV0LVJN7  Cell_Phones_and_Accessories        5      1   
1  0110400550  A1BJGDS0L1IO6I  Cell_Phones_and_Accessories        1      0   
2  0110400550   A1YX2RBMS1L9L  Cell_Phones_and_Accessories        5      1   
3  0110400550  A180NNPPKWCCU0  Cell_Phones_and_Accessories        5      1   
4  0110400550  A30P2CYOUYAJM8  Cell_Phones_and_Accessories        4      1   

   review_dt  helpful_up  helpful_tot  helpful_ratio  word_len  char_len  \
0 2013-01-13         4.0          4.0            1.0        24       115   
1 2013-01-30         0.0          3.0            0.0        32       158   
2 2012-11-22         0.0          0.0            NaN        23       115   
3 2013-07-18         3.0      

## Adım 6: Kaydedilen Parquet dosyasını özetle
Bu hücre, 'preprocessed_sample.parquet' dosyasını yükleyip satır/sütun sayısı, sütun tipleri, label ve kategori dağılımı, tarih aralığı ve null değerleri raporlar.

In [34]:
import os
import pandas as pd

pp_path = os.path.join(os.getcwd(), 'preprocessed_sample.parquet')
print('Dosya:', pp_path)
if not os.path.exists(pp_path):
    raise SystemExit('Parquet bulunamadı. Önce Adım 5 ile üretin.')

# Parquet'i yükle
try:
    df_pp = pd.read_parquet(pp_path)
except Exception as e:
    print('read_parquet başarısız:', e)
    print('PyArrow veya fastparquet kurulu mu?')
    raise

# Boyut / hafıza
rows, cols = df_pp.shape
mem_mb = df_pp.memory_usage(deep=True).sum() / (1024*1024)
print(f'Shape: {rows} satır x {cols} sütun | ~{mem_mb:.2f} MB')

# Sütunlar ve tipler
print('\nSütun tipleri:')
print(df_pp.dtypes)

# İlk satırlar
print('\nHead:')
print(df_pp.head(3))

# Label dağılımı
if 'label' in df_pp.columns:
    print('\nLabel dağılımı (adet / oran):')
    vc = df_pp['label'].value_counts(dropna=False)
    print(vc)
    print((vc/len(df_pp)).round(3))

# Kategori dağılımı
if 'category' in df_pp.columns:
    print('\nKategori dağılımı (ilk 10):')
    print(df_pp['category'].value_counts().head(10))

# Tarih aralığı
if 'review_dt' in df_pp.columns:
    print('\nTarih aralığı:')
    print('min:', df_pp['review_dt'].min(), '| max:', df_pp['review_dt'].max())

# Numerik özet
num_cols = df_pp.select_dtypes(include=['number']).columns.tolist()
if num_cols:
    print('\nSayısal kolon özetleri:')
    print(df_pp[num_cols].describe().T.head(15))

# Null değerler
print('\nNull değer sayıları:')
print(df_pp.isnull().sum())

Dosya: c:\Users\aadil\Desktop\YAP470\amazon-fake-review-detector\preprocessed_sample.parquet
Shape: 10000 satır x 14 sütun | ~5.76 MB

Sütun tipleri:
asin                     object
reviewerID               object
category                 object
overall                   int64
label                      int8
review_dt        datetime64[ns]
helpful_up              float32
helpful_tot             float32
helpful_ratio           float32
word_len                  int32
char_len                  int32
bang_cnt                  int16
qmark_cnt                 int16
text_clean               object
dtype: object

Head:
         asin      reviewerID                     category  overall  label  \
0  0110400550   A3HVRXV0LVJN7  Cell_Phones_and_Accessories        5      1   
1  0110400550  A1BJGDS0L1IO6I  Cell_Phones_and_Accessories        1      0   
2  0110400550   A1YX2RBMS1L9L  Cell_Phones_and_Accessories        5      1   

   review_dt  helpful_up  helpful_tot  helpful_ratio  word_len  char

# Machine Learning: Spam Detection Model

Şimdi temizlenmiş veriyi kullanarak spam detection modeli kuracağız:
1. **TF-IDF Vectorization**: Metni sayısal özelliklere dönüştür
2. **Train/Test Split**: Veriyi eğitim ve test setlerine böl  
3. **LogisticRegression**: Baseline model ile başla
4. **Evaluation**: Accuracy, precision, recall, F1-score

## Adım 7: TF-IDF Feature Extraction
Temizlenmiş metinleri sayısal özelliklere dönüştürüyoruz. TF-IDF hem kelime hem de karakter n-gramları kullanır.

In [ ]:
# Önce parquet'i yükle (eğer df_pp zaten mevcut değilse)
import os
import pandas as pd

if 'df_pp' not in globals():
    pp_path = os.path.join(os.getcwd(), 'preprocessed_sample.parquet')
    if not os.path.exists(pp_path):
        raise FileNotFoundError(f'Parquet dosyası bulunamadı: {pp_path}. Önce Adım 5 çalıştırın.')
    df_pp = pd.read_parquet(pp_path)

print('Yüklenen veri shape:', df_pp.shape)

# TF-IDF için gerekli importlar
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import numpy as np

# Text ve numerik özellikleri ayır
text_col = 'text_clean'
numeric_cols = [c for c in ['word_len', 'char_len', 'bang_cnt', 'qmark_cnt', 'helpful_ratio', 'overall'] 
                if c in df_pp.columns]

print('Text kolonu:', text_col)
print('Numerik kolonlar:', numeric_cols)

# Text kolonu kontrolü
if text_col not in df_pp.columns:
    raise ValueError(f"'{text_col}' kolonu bulunamadı! Önce Adım 4 çalıştırın.")

# TF-IDF parametreleri (spam detection için optimize edilmiş)
tfidf_word = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),        # unigram + bigram
    max_features=5000,         # top 5K word features
    min_df=3,                  # en az 3 dokümanda geçmeli
    max_df=0.8,               # max %80 dokümanda geçebilir
    stop_words='english',     # İngilizce stop words
    lowercase=True,
    token_pattern=r'\b[a-zA-Z]{2,}\b'  # en az 2 harfli kelimeler
)

tfidf_char = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),        # 3-5 karakter n-gramları
    max_features=2000,         # top 2K char features
    min_df=3,
    max_df=0.8,
    lowercase=True
)

# Text features
print('TF-IDF word features hesaplanıyor...')
X_text_word = tfidf_word.fit_transform(df_pp[text_col].fillna(''))
print('Word TF-IDF shape:', X_text_word.shape)

print('TF-IDF char features hesaplanıyor...')
X_text_char = tfidf_char.fit_transform(df_pp[text_col].fillna(''))
print('Char TF-IDF shape:', X_text_char.shape)

# Numerik features (standardize)
if numeric_cols:
    scaler = StandardScaler()
    X_numeric = scaler.fit_transform(df_pp[numeric_cols].fillna(0))
    print('Numeric features shape:', X_numeric.shape)
else:
    X_numeric = np.empty((len(df_pp), 0))

print('Feature extraction tamamlandı!')

Yüklenen veri shape: (10000, 14)
Text kolonu: text_clean
Numerik kolonlar: ['word_len', 'char_len', 'bang_cnt', 'qmark_cnt', 'helpful_ratio', 'overall']
TF-IDF word features hesaplanıyor...
Word TF-IDF shape: (10000, 5000)
TF-IDF char features hesaplanıyor...
Char TF-IDF shape: (10000, 2000)
Numeric features shape: (10000, 6)
Feature extraction tamamlandı!


## Adım 8: Train/Test Split ve Label Hazırlığı
Veriyi eğitim ve test setlerine bölüyoruz. Aynı reviewerID'nin farklı setlerde olmamasını sağlayarak data leakage'i önlüyoruz.

In [39]:
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
import pandas as pd

# Label kontrolü
if 'label' not in df_pp.columns:
    raise ValueError('Label kolonu bulunamadı! Adım 3 çalıştırıldı mı?')

y = df_pp['label'].values
print('Label dağılımı:')
print(pd.Series(y).value_counts())
print('Spam oranı:', pd.Series(y).mean())

# Tüm feature'ları birleştir (sparse matrices)
if X_numeric.shape[1] > 0:
    # Convert numeric to sparse for concatenation
    from scipy.sparse import csr_matrix
    X_numeric_sparse = csr_matrix(X_numeric)
    X_combined = hstack([X_text_word, X_text_char, X_numeric_sparse])
    feature_names = (['word_' + str(i) for i in range(X_text_word.shape[1])] + 
                    ['char_' + str(i) for i in range(X_text_char.shape[1])] + 
                    [f'num_{col}' for col in numeric_cols])
else:
    X_combined = hstack([X_text_word, X_text_char])
    feature_names = (['word_' + str(i) for i in range(X_text_word.shape[1])] + 
                    ['char_' + str(i) for i in range(X_text_char.shape[1])])

print('Birleşik feature matrix shape:', X_combined.shape)
print('Feature sayısı:', len(feature_names))

# Train/test split (reviewer-based split data leakage'i önlemek için)
if 'reviewerID' in df_pp.columns:
    # ReviewerID'ye göre split yap
    unique_reviewers = df_pp['reviewerID'].unique()
    train_reviewers, test_reviewers = train_test_split(
        unique_reviewers, test_size=0.2, random_state=42
    )
    
    train_mask = df_pp['reviewerID'].isin(train_reviewers)
    test_mask = df_pp['reviewerID'].isin(test_reviewers)
    
    # Sparse matrix için boolean mask'i numpy array'e çevir
    train_idx = train_mask.values  # pandas Series → numpy array
    test_idx = test_mask.values
    
    X_train, X_test = X_combined[train_idx], X_combined[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    print('Reviewer-based split yapıldı')
    print(f'Train reviewers: {len(train_reviewers)}, Test reviewers: {len(test_reviewers)}')
else:
    # Normal random split
    X_train, X_test, y_train, y_test = train_test_split(
        X_combined, y, test_size=0.2, random_state=42, stratify=y
    )
    print('Random split yapıldı')

print(f'Train shape: {X_train.shape}, y_train spam oranı: {y_train.mean():.3f}')
print(f'Test shape: {X_test.shape}, y_test spam oranı: {y_test.mean():.3f}')

Label dağılımı:
1    6347
0    3653
Name: count, dtype: int64
Spam oranı: 0.6347
Birleşik feature matrix shape: (10000, 7006)
Feature sayısı: 7006
Reviewer-based split yapıldı
Train reviewers: 7802, Test reviewers: 1951
Train shape: (8002, 7006), y_train spam oranı: 0.634
Test shape: (1998, 7006), y_test spam oranı: 0.636


## Adım 9: LogisticRegression Model Eğitimi
Spam detection için LogisticRegression baseline modeli eğitiyoruz. Class imbalance için `class_weight='balanced'` kullanıyoruz.

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import time

# LogisticRegression model (class imbalance için balanced)
lr_model = LogisticRegression(
    class_weight='balanced',   # imbalanced data için ağırlık ayarı
    max_iter=1000,            # yeterli iterasyon
    random_state=42,
    n_jobs=-1                 # paralel işlem
)

print('Model eğitimi başlıyor...')
start_time = time.time()

# Eğitim
lr_model.fit(X_train, y_train)

train_time = time.time() - start_time
print(f'Eğitim süresi: {train_time:.2f} saniye')

# Tahminler
print('Tahminler yapılıyor...')
y_train_pred = lr_model.predict(X_train)
y_test_pred = lr_model.predict(X_test)

# Probability scores (ROC-AUC için)
y_train_proba = lr_model.predict_proba(X_train)[:, 1]
y_test_proba = lr_model.predict_proba(X_test)[:, 1]

print('Model eğitimi ve tahmin tamamlandı!')

Model eğitimi başlıyor...
Eğitim süresi: 4.27 saniye
Tahminler yapılıyor...
Model eğitimi ve tahmin tamamlandı!
Eğitim süresi: 4.27 saniye
Tahminler yapılıyor...
Model eğitimi ve tahmin tamamlandı!


## Adım 10: Model Performans Değerlendirmesi
Train ve test setleri için accuracy, precision, recall, F1-score ve ROC-AUC metriklerini hesaplıyoruz.

In [41]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Train set performansı
train_acc = accuracy_score(y_train, y_train_pred)
train_prec = precision_score(y_train, y_train_pred)
train_rec = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)
train_auc = roc_auc_score(y_train, y_train_proba)

# Test set performansı  
test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred)
test_rec = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_proba)

print('=== MODEL PERFORMANSI ===')
print(f'{"Metrik":<15} {"Train":<10} {"Test":<10} {"Fark":<10}')
print('-' * 50)
print(f'{"Accuracy":<15} {train_acc:<10.4f} {test_acc:<10.4f} {abs(train_acc-test_acc):<10.4f}')
print(f'{"Precision":<15} {train_prec:<10.4f} {test_prec:<10.4f} {abs(train_prec-test_prec):<10.4f}')
print(f'{"Recall":<15} {train_rec:<10.4f} {test_rec:<10.4f} {abs(train_rec-test_rec):<10.4f}')
print(f'{"F1-Score":<15} {train_f1:<10.4f} {test_f1:<10.4f} {abs(train_f1-test_f1):<10.4f}')
print(f'{"ROC-AUC":<15} {train_auc:<10.4f} {test_auc:<10.4f} {abs(train_auc-test_auc):<10.4f}')

print('\n=== CONFUSION MATRIX (Test Set) ===')
cm = confusion_matrix(y_test, y_test_pred)
print('Predicted:   0 (Not Spam)  1 (Spam)')
print(f'Actual 0:    {cm[0,0]:<12} {cm[0,1]:<7}')
print(f'Actual 1:    {cm[1,0]:<12} {cm[1,1]:<7}')

print('\n=== CLASSIFICATION REPORT (Test Set) ===')
print(classification_report(y_test, y_test_pred, target_names=['Not Spam', 'Spam']))

# Overfitting kontrolü
train_test_diff = abs(train_f1 - test_f1)
if train_test_diff > 0.05:
    print(f'\n⚠️  UYARI: Train-test F1 farkı {train_test_diff:.4f} > 0.05, overfitting olabilir')
else:
    print(f'\n✅ İyi: Train-test F1 farkı {train_test_diff:.4f} < 0.05, overfitting düşük')

=== MODEL PERFORMANSI ===
Metrik          Train      Test       Fark      
--------------------------------------------------
Accuracy        1.0000     1.0000     0.0000    
Precision       1.0000     1.0000     0.0000    
Recall          1.0000     1.0000     0.0000    
F1-Score        1.0000     1.0000     0.0000    
ROC-AUC         1.0000     1.0000     0.0000    

=== CONFUSION MATRIX (Test Set) ===
Predicted:   0 (Not Spam)  1 (Spam)
Actual 0:    728          0      
Actual 1:    0            1270   

=== CLASSIFICATION REPORT (Test Set) ===
              precision    recall  f1-score   support

    Not Spam       1.00      1.00      1.00       728
        Spam       1.00      1.00      1.00      1270

    accuracy                           1.00      1998
   macro avg       1.00      1.00      1.00      1998
weighted avg       1.00      1.00      1.00      1998


✅ İyi: Train-test F1 farkı 0.0000 < 0.05, overfitting düşük


## Adım 11: Feature Importance Analizi
En önemli spam göstergesi kelime/özellikler hangileri? LogisticRegression katsayılarını inceliyoruz.

In [42]:
import numpy as np

# LogisticRegression katsayıları
coefficients = lr_model.coef_[0]

# Feature importance (mutlak değerler)
feature_importance = np.abs(coefficients)
sorted_idx = np.argsort(feature_importance)[::-1]

print('=== EN ÖNEMLİ FEATURES (Top 20) ===')
print(f'{"Rank":<5} {"Feature":<25} {"Coefficient":<12} {"Importance":<12}')
print('-' * 60)

for i, idx in enumerate(sorted_idx[:20]):
    if i < len(feature_names):
        feat_name = feature_names[idx]
        coef_val = coefficients[idx]
        importance = feature_importance[idx]
        
        # Feature tipini belirle
        if feat_name.startswith('word_'):
            # Word feature'dan gerçek kelimeyi çıkar
            word_idx = int(feat_name.split('_')[1])
            if word_idx < len(tfidf_word.get_feature_names_out()):
                actual_word = tfidf_word.get_feature_names_out()[word_idx]
                display_name = f'WORD: {actual_word}'
            else:
                display_name = feat_name
        elif feat_name.startswith('char_'):
            # Char feature'dan gerçek n-gramı çıkar  
            char_idx = int(feat_name.split('_')[1])
            if char_idx < len(tfidf_char.get_feature_names_out()):
                actual_char = tfidf_char.get_feature_names_out()[char_idx]
                display_name = f'CHAR: {actual_char}'
            else:
                display_name = feat_name
        else:
            display_name = feat_name
            
        print(f'{i+1:<5} {display_name:<25} {coef_val:<12.6f} {importance:<12.6f}')

# Spam vs Not Spam indicator katsayıları
print('\n=== SPAM vs NOT SPAM FEATURES ===')
print('Pozitif katsayı = SPAM indicator')
print('Negatif katsayı = NOT SPAM indicator')

spam_features = []
not_spam_features = []

for i, idx in enumerate(sorted_idx[:50]):  # top 50'yi incele
    if i < len(feature_names):
        feat_name = feature_names[idx]
        coef_val = coefficients[idx]
        
        if feat_name.startswith('word_'):
            word_idx = int(feat_name.split('_')[1])
            if word_idx < len(tfidf_word.get_feature_names_out()):
                actual_word = tfidf_word.get_feature_names_out()[word_idx]
                if coef_val > 0:
                    spam_features.append((actual_word, coef_val))
                else:
                    not_spam_features.append((actual_word, coef_val))

print('\nTop 10 SPAM kelimeler:')
for word, coef in sorted(spam_features, key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {word}: {coef:.4f}')
    
print('\nTop 10 NOT SPAM kelimeler:')
for word, coef in sorted(not_spam_features, key=lambda x: x[1])[:10]:
    print(f'  {word}: {coef:.4f}')

=== EN ÖNEMLİ FEATURES (Top 20) ===
Rank  Feature                   Coefficient  Importance  
------------------------------------------------------------
1     num_overall               12.150793    12.150793   
2     WORD: ok                  -0.387522    0.387522    
3     WORD: great               0.221093     0.221093    
4     WORD: good                0.199387     0.199387    
5     num_bang_cnt              0.198303     0.198303    
6     WORD: works               0.190480     0.190480    
7     CHAR: but                 -0.175405    0.175405    
8     CHAR: but                 -0.172204    0.172204    
9     CHAR:  but                -0.171759    0.171759    
10    CHAR:  but                -0.170550    0.170550    
11    CHAR: ut                  -0.158115    0.158115    
12    WORD: broke               -0.149833    0.149833    
13    CHAR:  bu                 -0.140862    0.140862    
14    CHAR:  not                -0.135660    0.135660    
15    CHAR:  not                -